<a href="https://colab.research.google.com/github/lahari600/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lahari600/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents the daily search and analytics performance of one content page for one client on one report date.

Time Window:
I will use a mid-panel month (such as March 2026) for feature engineering and model development. The final month will be kept as a test period to avoid data leakage.

In [23]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")
print(token[:10] + "...")

hf_RhOaMDQ...


In [24]:
!pip -q install datasets huggingface_hub pyarrow

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [26]:
from google.colab import userdata
from datasets import load_dataset

token = userdata.get("HF_TOKEN")

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=token,
    streaming=True
)

print(dataset)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_shards: 18
    })
})


In [27]:
from itertools import islice
import pandas as pd

sample = list(islice(dataset["train"], 1000))   # First 1000 rows
df = pd.DataFrame(sample)

print("Shape:", df.shape)
df.head()

Shape: (1000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields are historical search and analytics metrics that are available when making a refresh decision.

Label:
Refresh priority (whether a page should be refreshed first).

Context:
Report date, client ID and content ID identify each record.

Excluded:
Future information and outcome-based metrics are excluded because they are not available at decision time and would cause data leakage.

In [28]:
feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

label_field = "refresh_priority"

context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

excluded_fields = [
    "future performance metrics"
]

print("Feature Fields:", feature_fields)
print("Label:", label_field)
print("Context:", context_fields)
print("Excluded:", excluded_fields)

Feature Fields: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'scroll_events']
Label: refresh_priority
Context: ['report_date', 'client_hash_id', 'content_hash_id']
Excluded: ['future performance metrics']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [29]:
print(df[["report_date",
          "client_hash_id",
          "content_hash_id"]].head())

  report_date           client_hash_id           content_hash_id
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058
3  2025-01-27  client_9958f0a7ae1df715  content_c899aef92518c714
4  2025-01-27  client_9958f0a7ae1df715  content_c7c1d2e68d9d0964


In [30]:
print("Rows:", len(df))

print("Date Range:")
print(df["report_date"].min())
print(df["report_date"].max())

Rows: 1000
Date Range:
2025-01-27
2025-01-30


In [31]:
available = df[df["gsc_data_available"] == True]

print("Rows with GSC data:", len(available))

Rows with GSC data: 1000


In [32]:
features = df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "scroll_events"
    ]
]

features.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,30,0,3.833333,0,0
1,5,0,71.600000,0,0
2,1,0,34.000000,0,0
3,6,0,23.333333,0,0
4,5,0,17.800000,0,0


1. gsc_impressions
Available at decision time because historical search impressions already exist.

2. gsc_clicks
Available because past click data is collected daily.

3. gsc_avg_position
Available from previous search rankings.

4. ga4_sessions
Available because website analytics are collected before prediction.

5. scroll_events
Available because user engagement history already exists.

In [33]:
df["refresh_priority"] = (df["gsc_clicks"] < 5).astype(int)

df["leak_feature"] = df["refresh_priority"]

print(df[["refresh_priority",
          "leak_feature"]].head())

   refresh_priority  leak_feature
0                 1             1
1                 1             1
2                 1             1
3                 1             1
4                 1             1


The column "leak_feature" is created directly from the label.

A model trained with this feature would achieve unrealistically high accuracy because it already contains the answer.

This feature must be removed before training.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Limitation

This notebook uses only a streamed sample of the warehouse rather than the complete dataset. Therefore, the analysis demonstrates the workflow but does not represent the full production data.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Plain-language data contract
Three verification queries
Five features with availability explanations
Leakage demonstration and removal
Limitation statement